# T35 - 不同曲线投资顺序分析
## 第6章：可视化

### 概述
本章生成策略分析的可视化图表，包括：
1. 策略收益率对比图
2. 策略稳定性分析图
3. 各周期策略表现图
4. 累计收益曲线图

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import FontProperties
import matplotlib.gridspec as gridspec
import os
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    OUTPUT_DIR,
    CHART_COLORS,
    FIGURE_SIZE,
    TITLE_SIZE,
    LABEL_SIZE,
    TICK_SIZE
)

print(f"输出目录: {OUTPUT_DIR}")
print(f"图表尺寸: {FIGURE_SIZE}")

In [ ]:
# 3. 设置中文字体和全局样式

def setup_chinese_font():
    """设置中文字体"""
    font_paths = [
        r"C:\Windows\Fonts\msyh.ttc",
        r"C:\Windows\Fonts\simhei.ttf",
        r"C:\Windows\Fonts\simsun.ttc",
    ]
    
    for font_path in font_paths:
        try:
            font = FontProperties(fname=font_path)
            print(f"成功加载中文字体: {font_path}")
            plt.rcParams['axes.unicode_minus'] = False
            return font
        except:
            continue
    
    print("警告: 无法加载中文字体")
    return None

chinese_font = setup_chinese_font()

# 设置全局样式
plt.style.use('default')
plt.rcParams['figure.figsize'] = FIGURE_SIZE
plt.rcParams['axes.titlesize'] = TITLE_SIZE
plt.rcParams['axes.labelsize'] = LABEL_SIZE
plt.rcParams['xtick.labelsize'] = TICK_SIZE
plt.rcParams['ytick.labelsize'] = TICK_SIZE
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '--'

In [ ]:
# 4. 加载回测结果

def load_backtest_results():
    """加载回测结果"""
    results = {}
    dimensions = ['品种维度', '久期维度', '资质维度']
    
    for dimension in dimensions:
        file_path = os.path.join(OUTPUT_DIR, f'{dimension}回测结果.csv')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, encoding='utf-8-sig')
            results[dimension] = df
            print(f"加载 {dimension}: {len(df)} 条记录")
        else:
            print(f"警告: 文件不存在 - {file_path}")
    
    return results

backtest_results = load_backtest_results()

In [ ]:
# 5. 定义可视化类

class PortfolioVisualizer:
    """
    投资组合可视化类
    
    用于生成策略分析图表
    """
    
    def __init__(self, colors=None, fig_size=None, chinese_font=None):
        self.colors = colors or CHART_COLORS
        self.fig_size = fig_size or FIGURE_SIZE
        self.chinese_font = chinese_font
    
    def plot_strategy_comparison(self, df, dimension):
        """
        绘制策略对比图
        
        Parameters:
        -----------
        df : DataFrame
            回测结果数据
        dimension : str
            维度名称
        """
        plt.figure(figsize=self.fig_size)
        
        # 计算每个策略的平均表现
        strategy_means = df.groupby('strategy_sequence')['annual_return'].mean().sort_values(ascending=True)
        
        # 创建柱状图
        bars = plt.barh(range(len(strategy_means)), strategy_means.values, 
                       color=self.colors[:len(strategy_means)])
        
        # 设置y轴标签
        labels = [s[:30] + '...' if len(s) > 30 else s for s in strategy_means.index]
        if self.chinese_font:
            plt.yticks(range(len(strategy_means)), labels, fontproperties=self.chinese_font)
        else:
            plt.yticks(range(len(strategy_means)), labels)
        
        # 添加数值标签
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width, i, f'{width:.2%}', 
                    va='center', fontsize=TICK_SIZE)
        
        title = f'{dimension}策略年化收益率对比'
        if self.chinese_font:
            plt.title(title, fontproperties=self.chinese_font)
            plt.xlabel('年化收益率', fontproperties=self.chinese_font)
            plt.ylabel('策略顺序', fontproperties=self.chinese_font)
        else:
            plt.title(title)
            plt.xlabel('Annual Return')
            plt.ylabel('Strategy')
        
        plt.tight_layout()
        
        # 保存图表
        output_path = os.path.join(OUTPUT_DIR, f'{dimension}策略对比.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存: {output_path}")
        
        plt.show()
    
    def plot_strategy_stability(self, df, dimension):
        """
        绘制策略稳定性分析图
        
        Parameters:
        -----------
        df : DataFrame
            回测结果数据
        dimension : str
            维度名称
        """
        fig = plt.figure(figsize=(15, 6))
        gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1])
        
        # 1. 箱线图
        ax1 = plt.subplot(gs[0])
        
        # 准备数据
        strategy_names = df['strategy_sequence'].unique()
        data_for_boxplot = [df[df['strategy_sequence'] == s]['annual_return'].values for s in strategy_names]
        
        bp = ax1.boxplot(data_for_boxplot, patch_artist=True)
        
        # 设置箱线图颜色
        for i, box in enumerate(bp['boxes']):
            box.set(facecolor=self.colors[i % len(self.colors)])
        
        # 设置x轴标签
        labels = [s[:15] + '...' if len(s) > 15 else s for s in strategy_names]
        ax1.set_xticklabels(labels, rotation=45, ha='right')
        if self.chinese_font:
            for label in ax1.get_xticklabels():
                label.set_fontproperties(self.chinese_font)
            ax1.set_title('策略收益率分布', fontproperties=self.chinese_font)
            ax1.set_xlabel('策略顺序', fontproperties=self.chinese_font)
            ax1.set_ylabel('年化收益率', fontproperties=self.chinese_font)
        else:
            ax1.set_title('Return Distribution')
            ax1.set_xlabel('Strategy')
            ax1.set_ylabel('Annual Return')
        
        # 2. 密度图
        ax2 = plt.subplot(gs[1])
        for i, strategy in enumerate(strategy_names):
            strategy_data = df[df['strategy_sequence'] == strategy]['annual_return']
            if len(strategy_data) > 1:
                sns.kdeplot(data=strategy_data, ax=ax2, label=strategy[:20],
                           color=self.colors[i % len(self.colors)])
        
        if self.chinese_font:
            ax2.set_title('策略收益率密度分布', fontproperties=self.chinese_font)
            ax2.set_xlabel('年化收益率', fontproperties=self.chinese_font)
            ax2.set_ylabel('密度', fontproperties=self.chinese_font)
        else:
            ax2.set_title('Return Density')
            ax2.set_xlabel('Annual Return')
            ax2.set_ylabel('Density')
        
        if self.chinese_font:
            ax2.legend(prop=self.chinese_font, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        else:
            ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        
        plt.tight_layout()
        
        # 保存图表
        output_path = os.path.join(OUTPUT_DIR, f'{dimension}策略稳定性分析.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存: {output_path}")
        
        plt.show()
    
    def plot_market_cycle_analysis(self, df, dimension, cycles_df=None):
        """
        绘制不同周期的策略表现对比图
        
        Parameters:
        -----------
        df : DataFrame
            回测结果数据
        dimension : str
            维度名称
        cycles_df : DataFrame, optional
            周期标注数据
        """
        # 按周期分组计算每个策略的年化收益率
        cycle_performance = df.pivot_table(
            index='cycle_id', 
            columns='strategy_sequence', 
            values='annual_return',
            aggfunc='mean'
        )
        
        # 创建图表
        plt.figure(figsize=(15, 8))
        
        # 设置柱状图的位置
        n_strategies = len(cycle_performance.columns)
        n_cycles = len(cycle_performance.index)
        width = 0.8 / n_strategies
        
        # 绘制柱状图
        for i, strategy in enumerate(cycle_performance.columns):
            x = np.arange(n_cycles) + i * width - (n_strategies-1) * width/2
            strategy_label = strategy[:20] + '...' if len(strategy) > 20 else strategy
            plt.bar(x, cycle_performance[strategy], width, 
                   label=strategy_label,
                   color=self.colors[i % len(self.colors)])
        
        # 设置图表标签
        title = f'历史不同行情时期{dimension}年化收益率对比'
        if self.chinese_font:
            plt.title(title, fontproperties=self.chinese_font, fontsize=14, pad=20)
            plt.xlabel('观察期', fontproperties=self.chinese_font, fontsize=12)
            plt.ylabel('年化收益率', fontproperties=self.chinese_font, fontsize=12)
        else:
            plt.title('Strategy Returns by Period')
            plt.xlabel('Period')
            plt.ylabel('Annual Return')
        
        # 设置x轴刻度
        cycle_labels = [f'周期{i}' for i in cycle_performance.index]
        plt.xticks(np.arange(n_cycles), cycle_labels)
        if self.chinese_font:
            for label in plt.gca().get_xticklabels():
                label.set_fontproperties(self.chinese_font)
        
        # 设置y轴为百分比格式
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
        
        # 添加网格线
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        
        # 添加图例
        if self.chinese_font:
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', 
                      prop=self.chinese_font, fontsize=9, frameon=True)
        else:
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', 
                      fontsize=9, frameon=True)
        
        # 调整布局
        plt.tight_layout()
        
        # 保存图表
        output_path = os.path.join(OUTPUT_DIR, f'{dimension}策略收益率对比.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存: {output_path}")
        
        plt.show()
    
    def plot_cumulative_returns(self, df, dimension):
        """
        绘制累计收益曲线
        
        Parameters:
        -----------
        df : DataFrame
            回测结果数据
        dimension : str
            维度名称
        """
        plt.figure(figsize=self.fig_size)
        
        for i, strategy in enumerate(df['strategy_sequence'].unique()):
            strategy_data = df[df['strategy_sequence'] == strategy].sort_values('cycle_id')
            cumulative_returns = (1 + strategy_data['annual_return']).cumprod()
            
            strategy_label = strategy[:25] + '...' if len(strategy) > 25 else strategy
            plt.plot(range(len(cumulative_returns)), cumulative_returns,
                    label=strategy_label, linewidth=2, 
                    color=self.colors[i % len(self.colors)])
        
        if self.chinese_font:
            plt.title(f'{dimension}策略累计收益对比', fontproperties=self.chinese_font)
            plt.xlabel('观察期数', fontproperties=self.chinese_font)
            plt.ylabel('累计收益倍数', fontproperties=self.chinese_font)
            plt.legend(prop=self.chinese_font, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        else:
            plt.title(f'{dimension} Cumulative Returns')
            plt.xlabel('Period')
            plt.ylabel('Cumulative Return')
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        
        plt.grid(True, linestyle='--', alpha=0.7)
        
        plt.tight_layout()
        
        # 保存图表
        output_path = os.path.join(OUTPUT_DIR, f'{dimension}累计收益.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"图表已保存: {output_path}")
        
        plt.show()

# 创建可视化器
visualizer = PortfolioVisualizer(
    colors=CHART_COLORS,
    fig_size=FIGURE_SIZE,
    chinese_font=chinese_font
)

print("PortfolioVisualizer 类已定义")

In [ ]:
# 6. 生成所有图表

def generate_all_charts(backtest_results, visualizer):
    """生成所有维度的图表"""
    print("生成图表...")
    print("=" * 60)
    
    for dimension, df in backtest_results.items():
        print(f"\n{dimension}:")
        
        # 1. 策略对比图
        print("  1. 策略对比图")
        visualizer.plot_strategy_comparison(df, dimension)
        
        # 2. 策略稳定性分析图
        print("  2. 策略稳定性分析图")
        visualizer.plot_strategy_stability(df, dimension)
        
        # 3. 各周期策略表现图
        print("  3. 各周期策略表现图")
        visualizer.plot_market_cycle_analysis(df, dimension)
        
        # 4. 累计收益曲线图
        print("  4. 累计收益曲线图")
        visualizer.plot_cumulative_returns(df, dimension)
    
    print("\n所有图表生成完成！")

generate_all_charts(backtest_results, visualizer)

In [ ]:
# 7. 查看生成的图表文件

def list_generated_charts():
    """列出所有生成的图表文件"""
    print("生成的图表文件")
    print("=" * 60)
    
    chart_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png')]
    chart_files.sort()
    
    total_size = 0
    for f in chart_files:
        file_path = os.path.join(OUTPUT_DIR, f)
        size = os.path.getsize(file_path)
        total_size += size
        print(f"  - {f} ({size:,} bytes)")
    
    print(f"\n总计: {len(chart_files)} 个图表文件, {total_size:,} bytes")

list_generated_charts()

### 总结

本章完成了以下工作：
1. 设置了中文字体和全局图表样式
2. 加载了回测结果数据
3. 定义了 `PortfolioVisualizer` 可视化类
4. 生成了四种类型的图表：
   - 策略对比图（横向柱状图）
   - 策略稳定性分析图（箱线图+密度图）
   - 各周期策略表现图（分组柱状图）
   - 累计收益曲线图（折线图）

**输出文件（每个维度生成4张图）:**
- `品种维度策略对比.png`
- `品种维度策略稳定性分析.png`
- `品种维度策略收益率对比.png`
- `品种维度累计收益.png`
- `久期维度策略对比.png`
- `久期维度策略稳定性分析.png`
- `久期维度策略收益率对比.png`
- `久期维度累计收益.png`
- `资质维度策略对比.png`
- `资质维度策略稳定性分析.png`
- `资质维度策略收益率对比.png`
- `资质维度累计收益.png`

**下一步**: 运行 `07-报告汇总.ipynb` 生成最终分析报告。